In this notebook, I will fix the TD measurement and covariance matrices. I have a ground truth time-delay and the time-delay used in forecasting was mismatched. This is because there's an additional mass sheet assumed in the forecast, and I assumed a kappa_ext that I didn't keep track of, so without the kappa_ext, if I use the lens model I started with, the ground truth time-delay doesn't match the lens model predicted time-delay.

Collectively, the internal mass sheet and the external convergence act as scaling factors, so in this notebook, I am just going to scale the time-delay measurement mean and covariance by a scalar.

In [1]:
import numpy as np
import pandas as pd


In [2]:

df_truth = pd.read_csv("truth_metadata.csv")
df_truth

,main_deflector_parameters_center_x,main_deflector_parameters_center_y,main_deflector_parameters_dec_0,main_deflector_parameters_e1,main_deflector_parameters_e2,main_deflector_parameters_gamma,main_deflector_parameters_gamma1,main_deflector_parameters_gamma2,main_deflector_parameters_ra_0,main_deflector_parameters_theta_E,...,sigma_v_NIRSPEC_bin7_kmpersec,sigma_v_NIRSPEC_bin8_kmpersec,sigma_v_NIRSPEC_bin9_kmpersec,fpd01,fpd02,fpd03,Ddt_Mpc,td01,td02,td03
0,0.012326,-0.029470,0.0,-0.237521,-0.038308,2.113863,-0.023288,-0.000753,0.0,0.842323,...,168.904974,169.871742,167.141046,-0.386797,NaN,NaN,3643.354140,-43.841759,NaN,NaN
1,-0.009240,0.055357,0.0,-0.170785,-0.326370,1.736185,0.035391,-0.009625,0.0,0.971630,...,190.909387,196.680618,203.405935,-0.505634,-0.567742,-0.571985,3693.352030,-60.424795,-67.846857,-68.353976
2,-0.035690,-0.028720,0.0,-0.053444,-0.048492,2.017652,-0.004609,0.021804,0.0,0.277571,...,103.795959,100.074902,98.919730,-0.069209,NaN,NaN,9789.851900,-18.445097,NaN,NaN
3,-0.037636,0.012262,0.0,0.040738,0.106863,1.870207,-0.004262,0.006744,0.0,0.781619,...,162.109754,167.120645,169.307850,NaN,NaN,NaN,2120.107693,NaN,NaN,NaN
4,0.030015,-0.037275,0.0,0.004416,0.056147,2.122834,0.070019,0.033466,0.0,0.283372,...,79.722187,81.021617,84.137532,-0.149082,NaN,NaN,6053.992469,-26.201463,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
149,0.015969,-0.013192,0.0,-0.059482,0.034708,1.751160,0.045797,0.022826,0.0,0.241687,...,92.409485,98.485072,105.231421,-0.039930,NaN,NaN,3684.834211,-4.653655,NaN,NaN
150,0.005917,-0.002083,0.0,0.121605,0.047324,1.783938,-0.001539,0.023887,0.0,0.621044,...,102.038990,104.847943,110.479988,-0.010976,-0.021626,-0.070509,1152.533940,-0.348930,-0.687473,-2.241429
151,-0.037414,0.042149,0.0,0.186710,-0.252781,1.894661,0.010201,0.002586,0.0,0.434632,...,110.160882,105.221599,101.522219,-0.028550,-0.045422,-0.066613,2021.539013,-1.674040,-2.663371,-3.905896
152,0.027453,0.011626,0.0,-0.310230,-0.019298,1.499827,-0.006663,0.011140,0.0,0.350717,...,146.892703,150.312749,149.706854,NaN,NaN,NaN,3852.318101,NaN,NaN,NaN


In [3]:
padma_truth = pd.read_csv("deblended_time_delays.csv", index_col=0)
padma_truth

,dataset,subfolder,bh_mass,closest_image_separation,deflector_redshift,eddington_ratio,inclination_angle,kappa_0,kappa_1,kappa_star_0,...,joint_cov_matrix,joint_uncertainty,kappa_2,kappa_3,kappa_star_2,kappa_star_3,shear_2,shear_3,time_delays_2,time_delays_3
0,deblcdouble653,6062026,7.660872,1.589331,0.646613,0.159564,1.266604,0.251036,0.559314,0.024996,...,[[4.96962173]],2.229265,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,deblcquad134,6062026,7.478398,0.795852,0.614830,0.637345,80.421778,0.329351,0.503797,0.018185,...,[[ 4.44444711 2.8806435 2.88572172 -1.09965...,1.080505,0.981378,0.944239,0.168761,0.245074,0.705149,0.696731,55.792767,55.961130
2,deblcdouble91,6062026,7.848341,0.536035,1.063872,0.243742,43.946725,0.310465,0.900022,0.028779,...,[[9.04211342]],3.007011,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,deblcdouble98,6062026,6.237018,1.535654,0.352567,0.645192,84.245541,0.389121,0.925655,0.016439,...,[[6.11099549]],2.472043,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,deblcdouble23,6062026,8.642333,0.599191,0.917997,0.428223,62.363768,0.265130,2.305149,0.017521,...,[[20.87006705]],4.568377,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
149,deblcdouble469,6112026,7.867572,0.525255,0.663422,0.496451,82.834008,0.497123,1.040704,0.022355,...,[[11.79303649]],3.434099,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
150,deblcquad421,6112026,7.683792,0.666107,0.234840,0.146198,15.066448,0.534811,0.542758,0.052778,...,[[ 1.04791592 0.52264961 0.53102494 -0.47240...,0.537280,0.669100,0.780134,0.058140,0.094916,0.454870,0.528495,0.725716,2.337275
151,deblcquad285,6112026,8.605347,0.456092,0.388462,0.385005,19.430985,0.336422,0.386990,0.018854,...,[[ 0.80228424 0.4412373 0.45150051 -0.30557...,0.464164,0.772079,0.945740,0.082805,0.130515,0.706392,0.856304,2.407317,3.635546
152,deblctriple73,6112026,7.583521,0.368663,0.528276,0.544372,72.795570,0.524847,0.568953,0.021084,...,[[ 0.91860437 0.68477769 -0.19206036]\n [ 0.6...,0.570025,0.845978,NaN,0.070514,NaN,0.474188,NaN,2.278486,NaN


In [4]:
time_delay_ground_truth = padma_truth[['AB_true', 'AC_true', 'AD_true']]

In [5]:
time_delay_ground_truth_forecast = df_truth[['td01', 'td02', 'td03']]

$$ \rm td_{f} = td_{p} \times \alpha $$

### compute alpha

In [6]:
time_delay_ground_truth = time_delay_ground_truth.rename(columns={'AB_true':'td01', 'AC_true':'td02', 'AD_true':'td03'})

In [7]:
# time_delay_ground_truth_forecast# (time_delay_ground_truth)
time_delay_ground_truth.shape, time_delay_ground_truth_forecast.shape

((154, 3), (154, 3))

In [8]:
time_delay_ground_truth_forecast# (time_delay_ground_truth)

,td01,td02,td03
0,-43.841759,NaN,NaN
1,-60.424795,-67.846857,-68.353976
2,-18.445097,NaN,NaN
3,NaN,NaN,NaN
4,-26.201463,NaN,NaN
...,...,...,...
149,-4.653655,NaN,NaN
150,-0.348930,-0.687473,-2.241429
151,-1.674040,-2.663371,-3.905896
152,NaN,NaN,NaN


In [9]:
alpha=time_delay_ground_truth_forecast.combine(time_delay_ground_truth, lambda s1, s2: s1.div(s2), fill_value=np.nan).dropna(how='all')

In [34]:
alpha[alpha['td01']>-1.0]

,td01,td02,td03
6,-0.843961,NaN,NaN
8,-0.944750,NaN,NaN
9,-0.921420,NaN,NaN
10,-0.688941,-0.659582,-1.256577
12,-0.972251,NaN,NaN
...,...,...,...
142,-0.966412,NaN,NaN
143,-0.946129,NaN,NaN
145,-0.945657,NaN,NaN
147,-0.887738,NaN,NaN


### rescale means

In [11]:
import re
import numpy as np
td_meas_list=padma_truth[['joint_median']].loc[alpha.index]#.iloc[1, 0]
td_measurement_means_joint = pd.DataFrame()

def extract_first_three(s):
    if pd.isna(s):
        return pd.Series([np.nan, np.nan, np.nan])
    nums = re.findall(r'-?\d+\.?\d*(?:[eE][+-]?\d+)?', s)
    nums = [float(n) for n in nums[:3]]
    # pad with NaN if fewer than 3 values
    nums += [np.nan] * (3 - len(nums))
    return pd.Series(nums)

td_measurement_means_joint[['td01', 'td02', 'td03']] = td_meas_list['joint_median'].apply(extract_first_three)
scaled_means_joint = td_measurement_means_joint * alpha

In [12]:
td_meas_list=padma_truth[['i_median']].loc[alpha.index]#.iloc[1, 0]

td_measurement_means_i = pd.DataFrame()

def extract_first_three(s):
    if pd.isna(s):
        return pd.Series([np.nan, np.nan, np.nan])
    nums = re.findall(r'-?\d+\.?\d*(?:[eE][+-]?\d+)?', s)
    nums = [float(n) for n in nums[:3]]
    # pad with NaN if fewer than 3 values
    nums += [np.nan] * (3 - len(nums))
    return pd.Series(nums)

td_measurement_means_i[['td01', 'td02', 'td03']] = td_meas_list['i_median'].apply(extract_first_three)
scaled_means_i = td_measurement_means_i * alpha

### rescale covariance matrix

In [13]:
td_cov_joint = padma_truth[['joint_cov_matrix']].loc[alpha.index]

scaled_covs_joint = pd.DataFrame()

def extract_cov(s, max_dim=3):
    """Parse a printed numpy 2D array string, return top-left k x k block,
    where k = min(true matrix size, max_dim). No padding."""
    if pd.isna(s):
        return None

    nums = [float(n) for n in re.findall(r'-?\d+\.?\d*(?:[eE][+-]?\d+)?', s)]
    n = int(round(np.sqrt(len(nums))))
    if n * n != len(nums):
        raise ValueError(f"Parsed {len(nums)} numbers, not a perfect square: {s[:60]}...")

    full = np.array(nums).reshape(n, n)
    k = min(n, max_dim)
    return full[:k, :k]

def scale_row(row, alpha_df, cov_col='cov'):
    cov = row[cov_col]
    k = cov.shape[0]
    alpha_cols = ['td01', 'td02', 'td03'][:k]
    alpha_vals = alpha_df.loc[row.name, alpha_cols].to_numpy(dtype=float)
    return cov * np.outer(alpha_vals, alpha_vals)

td_cov_joint['cov'] = td_cov_joint['joint_cov_matrix'].apply(extract_cov)
scaled_covs_joint['cov_scaled'] = td_cov_joint.apply(lambda row: scale_row(row, alpha), axis=1)

In [14]:
td_cov_i = padma_truth[['i_cov_matrix']].loc[alpha.index].dropna(how='all')
alpha_i = alpha.loc[td_cov_i.index]
scaled_covs_i = pd.DataFrame()

def extract_cov(s, max_dim=3):
    """Parse a printed numpy 2D array string, return top-left k x k block,
    where k = min(true matrix size, max_dim). No padding."""
    if pd.isna(s):
        return None

    nums = [float(n) for n in re.findall(r'-?\d+\.?\d*(?:[eE][+-]?\d+)?', s)]
    n = int(round(np.sqrt(len(nums))))
    if n * n != len(nums):
        raise ValueError(f"Parsed {len(nums)} numbers, not a perfect square: {s[:60]}...")

    full = np.array(nums).reshape(n, n)
    k = min(n, max_dim)
    return full[:k, :k]

def scale_row(row, alpha_df, cov_col='cov'):
    cov = row[cov_col]
    k = cov.shape[0]
    alpha_cols = ['td01', 'td02', 'td03'][:k]
    alpha_vals = alpha_df.loc[row.name, alpha_cols].to_numpy(dtype=float)
    return cov * np.outer(alpha_vals, alpha_vals)

td_cov_i['cov'] = td_cov_i['i_cov_matrix'].apply(extract_cov)
scaled_covs_i['cov_scaled'] = td_cov_i.apply(lambda row: scale_row(row, alpha_i), axis=1)

### save to td_measurements file

In [15]:
scaled_means_i = scaled_means_i.dropna(how='all')#
scaled_covs_i['prec_scaled'] = scaled_covs_i['cov_scaled'].apply(lambda cov: np.linalg.inv(cov))
td_i = pd.merge(scaled_means_i, scaled_covs_i, left_index=True, right_index=True, how='inner')
td_i['catalog_idx'] = df_truth.loc[td_i.index, 'catalog_idx'].values
td_i['td01_true'] = df_truth.loc[td_i.index, 'td01'].values
td_i.to_csv("time_delay_i_measurements.csv", index=False)

In [16]:
scaled_means_joint = scaled_means_joint.dropna(how='all')#
scaled_covs_joint['prec_scaled'] = scaled_covs_joint['cov_scaled'].apply(lambda cov: np.linalg.inv(cov))
td_joint = pd.merge(scaled_means_joint, scaled_covs_joint, left_index=True, right_index=True, how='inner')
td_joint['catalog_idx'] = df_truth.loc[td_joint.index, 'catalog_idx'].values
td_joint['td01_true'] = df_truth.loc[td_joint.index, 'td01'].values
td_joint.to_csv("time_delay_joint_measurements.csv", index=False)

In [37]:
alpha.loc[6, 'td01']**2 * td_cov_joint.loc[6, 'cov'][0]#*alpha

array([0.50270296])

In [38]:
td_cov_joint.loc[6, 'cov'][0]

array([0.70577539])

In [32]:
td_cov_joint.iloc[0, 1][0]

array([4.96962173])

In [33]:
td_cov_joint.iloc[0, 1][0] * np.outer(alpha.iloc[0, 0], alpha.iloc[0, 0])

array([[6.36416239]])

#### sanity checks

In [76]:
print(td_cov_joint.iloc[1, 0])

[[ 4.44444711  2.8806435   2.88572172 -1.09965131 -1.0336608   0.21907092]
 [ 2.8806435   8.64976156  1.46744238  4.78475644 -1.12465996 -6.09811986]
 [ 2.88572172  1.46744238  8.544993   -1.25779923  4.77875042  6.06861689]
 [-1.09965131  4.78475644 -1.25779923  6.06700477 -0.10165908 -5.76512834]
 [-1.0336608  -1.12465996  4.77875042 -0.10165908  6.26856221  5.78832973]
 [ 0.21907092 -6.09811986  6.06861689 -5.76512834  5.78832973 13.41408385]]


In [77]:
print(td_cov_joint.iloc[1, 1])

[[4.44444711 2.8806435  2.88572172]
 [2.8806435  8.64976156 1.46744238]
 [2.88572172 1.46744238 8.544993  ]]


In [80]:
print(scaled_covs_joint.iloc[1, 0])

[[ 6.5535241   4.25373003  4.28016326]
 [ 4.25373003 12.7910971   2.17966723]
 [ 4.28016326  2.17966723 12.74871246]]


In [79]:

scaled_covs_joint

,cov_scaled
0,[[6.364162385219097]]
1,"[[6.5535240996410895, 4.253730034805673, 4.280..."
2,[[9.435941069620725]]
4,[[21.02403840905375]]
6,[[0.5027029565851489]]
...,...
148,[[4.628489343323465]]
149,[[12.2932303439592]]
150,"[[0.932154585807243, 0.46696096962777417, 0.48..."
151,"[[0.9829089921995458, 0.5403350902571469, 0.53..."
